In [ ]:
%pip install datasets

import os
import pandas as pd
import torch
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset

# Optional HF dataset iterator for efficient batched inference
try:
    from datasets import Dataset
except ImportError as exc:
    raise ImportError(
        "Please install 'datasets' in your environment before running this notebook."
    ) from exc

# Config
KAGGLE_INPUT_FILE = "/kaggle/input/new-processed/Processed_Reviews.csv"
INPUT_FILE = "data/processed/Processed_Reviews.csv"
BATCH_SIZE = 32
MAX_TEXT_LEN = 256
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
MODEL_REVISION = "daefdd1f6ae931839bce4d0f3db0a1a4265cd50f"
PIPELINE_DEVICE = 0 if torch.cuda.is_available() else -1


def build_output_path(input_file):
    input_abs = os.path.abspath(input_file)
    input_dir = os.path.dirname(input_abs)
    base_name = os.path.splitext(os.path.basename(input_abs))[0]
    out_name = f"{base_name}_with_Sentiment.csv"

    if os.path.isdir(input_dir) and os.access(input_dir, os.W_OK):
        return os.path.join(input_dir, out_name)

    for candidate_dir in ["/kaggle/working", os.getcwd()]:
        try:
            os.makedirs(candidate_dir, exist_ok=True)
            if os.access(candidate_dir, os.W_OK):
                return os.path.join(candidate_dir, out_name)
        except Exception:
            pass

    return out_name


OUTPUT_FILE = build_output_path(INPUT_FILE)

# Load data
df = pd.read_csv(INPUT_FILE)

# Build combined text from Title + Text
if "Title" not in df.columns and "Text" not in df.columns:
    raise ValueError("Input must contain at least one of: 'Title' or 'Text'.")

title = df["Title"].fillna("").astype(str) if "Title" in df.columns else pd.Series([""] * len(df))
text = df["Text"].fillna("").astype(str) if "Text" in df.columns else pd.Series([""] * len(df))
combined_text = (title + " " + text).str.strip()

# Prepare valid rows
valid_indices = [i for i, t in enumerate(combined_text.tolist()) if isinstance(t, str) and t.strip()]
valid_texts = [combined_text.iloc[i] for i in valid_indices]

label_map = {"LABEL_0": "NEGATIVE", "LABEL_1": "NEUTRAL", "LABEL_2": "POSITIVE"}
predicted = [None] * len(combined_text)

# Load model with pinned revision for reproducibility
sentiment_pipe = pipeline(
    "text-classification",
    model=MODEL_NAME,
    revision=MODEL_REVISION,
    device=PIPELINE_DEVICE,
)

if valid_texts:
    infer_ds = Dataset.from_dict({"text": valid_texts})
    outputs = sentiment_pipe(
        KeyDataset(infer_ds, "text"),
        truncation=True,
        max_length=MAX_TEXT_LEN,
        batch_size=BATCH_SIZE,
    )

    for idx, out in zip(valid_indices, outputs):
        raw_label = str(out.get("label", "")).strip().upper()
        predicted[idx] = label_map.get(raw_label, raw_label if raw_label else None)

# Append only one new column and save
df["Sentiment"] = predicted
df.to_csv(OUTPUT_FILE, index=False)

print("Saved output:", OUTPUT_FILE)
print("Final shape:", df.shape)
print(df[["Sentiment"]].head())